In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]='4'
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import pandas as pd
import numpy as np
import scipy.signal as signal
import scipy.interpolate as interp
from tqdm import tqdm
import matplotlib.pyplot as plt

In [4]:
# --- 2. GT 신호 필터링용 함수 ---
def butter_bandpass_filter(data, lowcut, highcut, fs, order=2):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    y = signal.filtfilt(b, a, data)
    return y

def z_norm(data):
    return (data - np.mean(data, axis = 0)) / (np.std(data, axis = 0) + 1e-8)


def z_norm_vid_out_of_bounds_bg(win_video_interpolated, bg_fill_value=-3.0):
    """
    유저의 관점 적용: 
    0.0 ~ 1.0 범위를 벗어난 외삽 노이즈와 원래 배경(0.0)을 모두 배경으로 취급하여 
    클리핑 없이 정규화 및 마스킹을 수행하는 함수
    """
    # 1. 유효 픽셀 및 배경 위치 파악 (클리핑 안 함)
    # 피부 픽셀의 조건: 1e-5 보다 크면서 1.0 이하인 값만 진짜 피부로 인정
    is_valid = (win_video_interpolated > 1e-5) & (win_video_interpolated <= 1.0)
    
    # 위 조건에 맞지 않는 모든 픽셀(원래 배경 0.0 + 음수 오버슈팅 + 1.0 초과 오버슈팅)은 배경으로 간주
    is_background = ~is_valid
    
    # 2. 유효 픽셀(조건을 만족하는 피부)만 모아서 Z-norm 통계량 계산
    valid_pixels = win_video_interpolated[is_valid]
    
    # 만약 유효 픽셀이 하나도 없다면 전체를 배경값으로 반환
    if len(valid_pixels) == 0:
        return np.full_like(win_video_interpolated, bg_fill_value)
        
    global_mean = np.mean(valid_pixels)
    global_std = np.std(valid_pixels) + 1e-8
    
    # 3. 전체 비디오에 대해 Z-norm 수행 (클리핑되지 않은 원본 사용)
    # 어차피 배경 영역은 아래에서 덮어씌워지므로, 연산 편의를 위해 전체에 적용
    norm_video = (win_video_interpolated - global_mean) / global_std
    
    # 4. 배경 위치(원래 배경 + 튀는 외삽 값)에 -3.0 덮어쓰기
    norm_video[is_background] = bg_fill_value 
    
    return norm_video


# --- 3. 핵심 윈도우 처리 및 보간 함수 ---
def process_video_and_gt_windows(video_data, gt_ppg, sub_valid, window_size=900, max_gap_threshold=30):
    n_samples = len(video_data)
    n_windows = n_samples // window_size
    
    valid_vid_list = []
    valid_gt_list = []
    discarded_indices = []

    for i in range(n_windows):
        start_idx = i * window_size
        end_idx = start_idx + window_size
        
        win_video = video_data[start_idx:end_idx].copy()
        win_gt = gt_ppg[start_idx:end_idx].copy()
        win_valid = sub_valid[start_idx:end_idx].copy()
        
        # 결손(Gap) 길이 체크
        is_invalid = ~win_valid
        if np.any(is_invalid):
            condition = np.concatenate(([0], is_invalid, [0]))
            idx = np.flatnonzero(condition[1:] != condition[:-1])
            gap_lengths = idx[1::2] - idx[::2]
            max_gap = gap_lengths.max() if len(gap_lengths) > 0 else 0
        else:
            max_gap = 0
            
        # 임계치 30프레임(1초) 이상 시 폐기
        if max_gap >= max_gap_threshold:
            discarded_indices.append(i)
            continue
            
        try:
            # 보간(Interpolation) 수행
            if max_gap > 0:
                valid_idx = np.where(win_valid)[0]
                all_idx = np.arange(window_size)
                
                if len(valid_idx) > 1:
                    # ✅ 수정된 부분: 다차원 영상 Linear 보간
                    linear_interpolator = interp.interp1d(
                        valid_idx, 
                        win_video[valid_idx], 
                        kind='linear', 
                        axis=0, 
                        fill_value='extrapolate'
                    )
                    win_video = linear_interpolator(all_idx)

                else:
                    discarded_indices.append(i)
                    continue
            
            # --- 영상 전처리 ---
            win_video_norm = z_norm_vid_out_of_bounds_bg(win_video)
            
            # --- GT 전처리 ---
            detrend_gt = signal.detrend(win_gt)
            filter_gt = butter_bandpass_filter(detrend_gt, 0.75, 2.5, fs_video, order=2)
            win_gt_norm = z_norm(filter_gt)
            
            # 유효 리스트에 추가 (정규화된 영상으로 저장!)
            valid_vid_list.append(win_video_norm)
            valid_gt_list.append(win_gt_norm)
            
        except Exception as e:
            print(f'Window {i} Error : {e}')
            discarded_indices.append(i)

    return valid_vid_list, valid_gt_list, discarded_indices


In [5]:
shift_frames = 400
fs_video = 30.0
fs_ppg = 500.0

# --- 1. 설정 및 파라미터 ---
people = ['A001_20250609', 'A002_20250609', 'A004_20250610', 'A005_20250611', 'A006_20250611', 
          'A007_20250612', 'A008_20250612', 'B001_20250623', 'B002_20250623', 'B003_20250624', 
          'B004_20250624', 'B005_20250625', 'B008_20250627', 'C001_20250701', 'C002_20250703', 'C003_20250702']

people_ppg = sorted([k.replace('_20', '_') for k in people])

preprocess_result_dir = '/nasdata4/2yhy/rPPG_code/rPPG_datasets/raw_data/CNA_members/KM_preprocess_data/preprocess_result'
ppg_cut_dir = '/nasdata4/2yhy/rPPG_code/rPPG_datasets/raw_data/CNA_members/KM_preprocess_data/ppg_cut_final' # 실제 GT 경로로 수정 필요
final_save_dir = '/nasdata4/2yhy/rPPG_code/rPPG_datasets/raw_data/CNA_members/KM_preprocess_data/data_linear_all_background' # 최종 저장 폴더 (미리 생성 필요)

os.makedirs(final_save_dir, exist_ok=True)



# --- 4. 메인 파이프라인 ---
for subject_id, ppg_id in zip(people, people_ppg):
    print(f"\n--- Processing {subject_id} ---")
    
    try:
        # 1. 이전 단계에서 만든 H5 비디오 데이터 로드
        h5_video_path = os.path.join(preprocess_result_dir, f'registration_video_[{subject_id}].h5')
        with h5py.File(h5_video_path, 'r') as f_v:
            video_data = f_v['nir_uv_data'][:]
            sub_valid = f_v['valid_mask'][:]
            
        # 2. GT PPG CSV 로드 (이 경로는 실제 환경에 맞춰 수정해 주세요)
        gt_csv_path = os.path.join(ppg_cut_dir, f'{ppg_id}.csv')
        ppg_cut = pd.read_csv(gt_csv_path).values.flatten()
        
        # 3. 싱크 조절 (400프레임 지연 반영)
        ppg_shift_samples = int(round((shift_frames / fs_video) * fs_ppg))
        
        ppg_trimmed = ppg_cut[ppg_shift_samples:]
        video_trimmed = video_data[:-shift_frames]
        sub_valid_trimmed = sub_valid[:-shift_frames]
        
        # GT 다운샘플링
        ppg_downsampled = signal.resample(ppg_trimmed, len(video_trimmed))
        
        # 4. 윈도우 분할 및 보간
        valid_vid_list, valid_gt_list, discarded = process_video_and_gt_windows(
            video_data=video_trimmed,
            gt_ppg=ppg_downsampled,
            sub_valid=sub_valid_trimmed,
            window_size=900,
            max_gap_threshold=30
        )
        
        if len(valid_vid_list) == 0:
            print(f"[{subject_id}] 유효한 윈도우가 없습니다. 건너뜁니다.")
            continue
            
        # 5. Array 형태로 묶기 (DataLoader 최적화)
        video_array = np.stack(valid_vid_list) # Shape: (유효 윈도우 수, 900, 64, 64)
        gt_array = np.stack(valid_gt_list)     # Shape: (유효 윈도우 수, 900)
        
        # 6. 최종 H5 파일 저장
        final_h5_path = os.path.join(final_save_dir, f'final_dataset_{subject_id}.h5')
        with h5py.File(final_h5_path, 'w') as f_out:
            f_out.create_dataset('video', data=video_array, chunks=True, compression='gzip', compression_opts=4)
            f_out.create_dataset('gt_ppg', data=gt_array, chunks=True, compression='gzip', compression_opts=4)
            
        print(f"[{subject_id}] 완료! 총 {len(valid_vid_list)}개 윈도우 저장됨. (폐기: {len(discarded)}개)")
        print(f"  -> Video Shape: {video_array.shape}, GT Shape: {gt_array.shape}")
        
    except Exception as e:
        print(f"[{subject_id}] 처리 중 오류 발생: {e}")


--- Processing A001_20250609 ---
[A001_20250609] 완료! 총 128개 윈도우 저장됨. (폐기: 1개)
  -> Video Shape: (128, 900, 64, 64), GT Shape: (128, 900)

--- Processing A002_20250609 ---
[A002_20250609] 완료! 총 130개 윈도우 저장됨. (폐기: 6개)
  -> Video Shape: (130, 900, 64, 64), GT Shape: (130, 900)

--- Processing A004_20250610 ---
[A004_20250610] 완료! 총 135개 윈도우 저장됨. (폐기: 1개)
  -> Video Shape: (135, 900, 64, 64), GT Shape: (135, 900)

--- Processing A005_20250611 ---
[A005_20250611] 완료! 총 85개 윈도우 저장됨. (폐기: 58개)
  -> Video Shape: (85, 900, 64, 64), GT Shape: (85, 900)

--- Processing A006_20250611 ---
[A006_20250611] 완료! 총 125개 윈도우 저장됨. (폐기: 0개)
  -> Video Shape: (125, 900, 64, 64), GT Shape: (125, 900)

--- Processing A007_20250612 ---
[A007_20250612] 완료! 총 88개 윈도우 저장됨. (폐기: 55개)
  -> Video Shape: (88, 900, 64, 64), GT Shape: (88, 900)

--- Processing A008_20250612 ---
[A008_20250612] 완료! 총 76개 윈도우 저장됨. (폐기: 63개)
  -> Video Shape: (76, 900, 64, 64), GT Shape: (76, 900)

--- Processing B001_20250623 ---
[B001_

In [6]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]='4'
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import torch
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader
import numpy as np
import glob
import sys
from tqdm import tqdm 


# data loading
class rPPGDataset(Dataset) :
    def __init__(self, h5_file_paths) : 
        self.data_info = []

        for path in h5_file_paths :
            with h5py.File(path, 'r', locking=False) as f :
                num_windows = f['video'].shape[0]
                for i in range(num_windows) :
                    self.data_info.append((path, i))
        
    def __len__(self) : ## len(train_dataset) 실행시 train_dataset.__len__() 이 실행됨
        return(len(self.data_info))

    def __getitem__(self, idx) :  ## train_dataset[0] 실행시 train_dataset.__getitem__(0) 이 실행됨
        file_path, window_idx = self.data_info[idx]

        with h5py.File(file_path, 'r', locking=False) as f:
            video = f['video'][window_idx]
            gt = f['gt_ppg'][window_idx]

        video = np.expand_dims(video, axis = 0)

        video_tensor = torch.tensor(video, dtype = torch.float32)
        gt_tensor = torch.tensor(gt, dtype = torch.float32)

        return video_tensor, gt_tensor


data_dir = '/nasdata4/2yhy/rPPG_code/rPPG_datasets/raw_data/CNA_members/KM_preprocess_data/data_linear_all_background'
all_files = sorted(glob.glob(os.path.join(data_dir, 'final_dataset_*.h5')))
all_files = [os.path.join(data_dir, u) for u in sorted(os.listdir(data_dir)) if u.endswith('.h5') ]

train_files = all_files[4:]
valid_files = all_files[2:4]
test_files = all_files[:2]

train_dataset = rPPGDataset(train_files)
valid_dataset = rPPGDataset(valid_files)
test_dataset = rPPGDataset(test_files)

batch_size = 8

train_loader = DataLoader(train_dataset, batch_size = batch_size , shuffle = True, num_workers =4)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

for inputs, targets in train_loader:
    print("\n[DataLoader Test]")
    print(f"Input Video Batch Shape: {inputs.shape}")  # 예상: [4, 1, 900, 64, 64]
    print(f"Target GT Batch Shape: {targets.shape}")   # 예상: [4, 900]
    break

            
        
# =================================================================
# 1. Resnet skip connection 기본 블록
# =================================================================

class ResidualBlock3D(nn.Module) :
    def __init__(self, in_channels, out_channels, spatial_stride = 1) :
        super().__init__()

        stride = (1, spatial_stride, spatial_stride) # 1자리에 들어가는 파라미터가 T축 압축을 결정 1로 설정했으므로 T 차원에서의 변화는 없음

        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size =(3,3,3), stride = stride, padding = 1, bias =False) # kernel_size : (T,H,W) (T: 시간축에서 평균내는 프레임의 수)  /  input shape : (900,in_channels,H,W) > output shape : (900, out_channels, )
        self.bn1 = nn.BatchNorm3d(out_channels)
        self.relu = nn.ReLU(inplace = True)

        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size = (3,3,3), stride = 1, padding = 1, bias = False)
        self.bn2 = nn.BatchNorm3d(out_channels) 

        self.downsample = None 
        if in_channels != out_channels or spatial_stride != 1 : ## input값이 residual block 값과 shape달라질 시 수행
            self.downsample = nn.Sequential(nn.Conv3d(in_channels, out_channels, kernel_size = 1, stride = stride, bias = False) ,
                                            nn.BatchNorm3d(out_channels))  

    def forward(self, x) :
        identity = x
        if self.downsample is not None :
            identity = self.downsample(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        out += identity  # 스킵 커넥션 (ResNet의 핵심)
        out = self.relu(out)
        
        return out


# =====================================================================
# 2. rPPG 전용 3D-ResNet 메인 네트워크
# =====================================================================
class rPPG_3D_ResNet(nn.Module):
    def __init__(self):
        super(rPPG_3D_ResNet, self).__init__()
        
        # [Step 1] 초기 특징 추출
        # 입력: [B, 1, 900, 64, 64] (NIR 1채널) -> 출력: [B, 16, 900, 64, 64]
        self.conv1 = nn.Conv3d(1, 16, kernel_size=(1, 5, 5), stride=1, padding=(0, 2, 2), bias=False)
        self.bn1 = nn.BatchNorm3d(16)
        self.relu = nn.ReLU(inplace=True)
        
        # [Step 2] ResNet 블록 통과 (공간 해상도만 점진적으로 절반씩 축소)
        # 시간축 900은 그대로 유지됩니다.
        self.layer1 = ResidualBlock3D(16, 32, spatial_stride=2) # -> [B, 32, 900, 32, 32]
        self.layer2 = ResidualBlock3D(32, 64, spatial_stride=2) # -> [B, 64, 900, 16, 16]
        self.layer3 = ResidualBlock3D(64, 64, spatial_stride=2) # -> [B, 64, 900, 8, 8]
        
        # [Step 3] 공간 평균 풀링 (Spatial Global Average Pooling)
        # 8x8 얼굴 공간의 특징들을 하나로 평균 내어 순수한 시간의 흐름만 남깁니다.
        # -> [B, 64, 900, 1, 1]
        self.adaptive_pool = nn.AdaptiveAvgPool3d((None, 1, 1))
        
        self.dropout = nn.Dropout3d(p=0.5)
        # [Step 4] 1D 맥파(rPPG) 파형 변환용 Regressor (1x1x1 Conv)
        # 64가닥의 단서들을 조합하여 최종 1개의 파형으로 만듭니다.
        # -> [B, 1, 900, 1, 1]
        self.regressor = nn.Conv3d(64, 1, kernel_size=1)

    def forward(self, x):
        # 1. 초기 컨볼루션
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        
        # 2. ResNet 블록 통과
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        
        # 3. 공간 뭉개기 및 1채널 압축
        x = self.adaptive_pool(x) 
        x = self.dropout(x)
        x = self.regressor(x) 
        
        # 4. 불필요한 차원(껍데기) 제거: [B, 1, 900, 1, 1] -> [B, 900]
        x = x.squeeze(-1).squeeze(-1).squeeze(1) 
        
        return x

class NegCorLoss(nn.Module) :
    def __init__(self) :
        super().__init__()
        
    def forward(self, preds, targets) :
        preds_norm = preds - torch.mean(preds, dim = -1, keepdim = True)
        targets_norm = targets - torch.mean(targets, dim = -1, keepdim = True)

        correlation =  torch.sum(preds_norm * targets_norm, dim = -1) / (torch.sqrt(torch.sum(preds_norm ** 2, dim=-1)) * torch.sqrt(torch.sum(targets_norm ** 2, dim=-1)) + 1e-8 )
        return  torch.mean(1 - correlation)

# 1. validate 함수 수정 [cite: 32, 33]
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for inputs, targets in loader: 
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            running_loss += loss.item()
            
    return running_loss / len(loader)

# 2. train_one_epoch 함수 수정 [cite: 29, 30]
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()

    return running_loss / len(loader)

# 설정 부분은 그대로 유지
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = rPPG_3D_ResNet().to(device) 
criterion = NegCorLoss()
lr_ = 1e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=lr_)
num_epochs = 200

best_val_loss = float('inf')
main_pbar = tqdm(range(num_epochs), desc="Total Epochs")

train_loss_list = []
valid_loss_list = []

for epoch in main_pbar:
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, valid_loader, criterion, device)
    
    train_loss_list.append(train_loss)
    valid_loss_list.append(val_loss)

    tqdm.write(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f'best_rppg_model_lnbg:{str(lr_)}_epoch:{str(epoch)}.pth')



[DataLoader Test]
Input Video Batch Shape: torch.Size([8, 1, 900, 64, 64])
Target GT Batch Shape: torch.Size([8, 900])


Total Epochs:   0%|          | 1/200 [02:41<8:54:11, 161.06s/it]

Epoch [1/200] Train Loss: 1.0009 | Val Loss: 0.9967


Total Epochs:   1%|          | 2/200 [05:07<8:24:05, 152.75s/it]

Epoch [2/200] Train Loss: 0.9864 | Val Loss: 0.9985


...

Epoch [198/200] Train Loss: 0.2160 | Val Loss: 1.0039


Total Epochs: 100%|█████████▉| 199/200 [8:14:13<02:34, 154.41s/it]

Epoch [199/200] Train Loss: 0.2242 | Val Loss: 1.0019


Total Epochs: 100%|██████████| 200/200 [8:16:45<00:00, 149.03s/it]

Epoch [200/200] Train Loss: 0.2138 | Val Loss: 1.0009
